# 006: Gradient Descent Optimization — Loss surface geometry, learning rate effects, and convergence analysis

## 1. GRADIENT DESCENT DERIVATION
- **Objective**: Minimize loss $\mathcal{L}(\theta)$ by iteratively updating parameters in the direction of steepest descent:
  $$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t)$$
- **Learning Rate $\eta$**: Controls step size. Too large → overshoots minima (divergence). Too small → slow convergence.

## 2. VARIANTS
- **Batch GD**: Computes gradient over the entire training set. Stable but slow for large datasets.
- **Stochastic GD (SGD)**: Uses a single random sample per update. Noisy but fast; noise can help escape local minima.
- **Mini-batch GD**: Compromise — uses batches of $B$ samples. Standard in practice ($B = 32, 64, 128$).

## 3. LOSS SURFACE GEOMETRY
- Neural network loss surfaces are high-dimensional and non-convex.
- **Saddle Points** (not local minima) are the primary obstacle in high dimensions.
- **Ravines**: Regions where the surface curves much more steeply in one dimension than another, causing oscillations.

---


In [ ]:
import numpy as np

# ── 1D Gradient Descent Visualization ──
def f(x):
    """Non-convex loss function: f(x) = x^4 - 3x^2 + 2."""
    return x**4 - 3*x**2 + 2

def f_grad(x):
    """Analytical gradient: f'(x) = 4x^3 - 6x."""
    return 4*x**3 - 6*x

def gradient_descent_1d(start, lr, n_steps):
    """Run gradient descent on f(x), tracking the trajectory."""
    x = start
    trajectory = [(x, f(x))]
    for step in range(n_steps):
        grad = f_grad(x)
        x = x - lr * grad
        trajectory.append((x, f(x)))
    return trajectory



In [ ]:
print("--- 1D Gradient Descent: Learning Rate Comparison ---")
for lr in [0.01, 0.05, 0.1]:
    traj = gradient_descent_1d(start=-0.5, lr=lr, n_steps=30)
    final_x, final_loss = traj[-1]
    print(f"  lr={lr:.2f} → Final x={final_x:+.4f}, Loss={final_loss:.4f} ({len(traj)-1} steps)")



In [ ]:
# ── Mini-batch SGD on synthetic 2D classification ──
np.random.seed(7)
# Generate 2-class spiral data
N = 100  # samples per class
X0 = np.random.randn(N, 2) * 0.5 + np.array([1, 1])
X1 = np.random.randn(N, 2) * 0.5 + np.array([-1, -1])
X = np.vstack([X0, X1]).T  # (2, 200)
Y = np.hstack([np.zeros(N), np.ones(N)]).reshape(1, -1)  # (1, 200)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Simple logistic regression via mini-batch SGD
W = np.random.randn(1, 2) * 0.01
b = 0.0
lr = 0.1
batch_size = 32
losses = []

for epoch in range(50):
    # Shuffle data
    perm = np.random.permutation(2 * N)
    X_shuf = X[:, perm]
    Y_shuf = Y[:, perm]
    epoch_loss = 0.0
    
    for i in range(0, 2 * N, batch_size):
        Xb = X_shuf[:, i:i+batch_size]
        Yb = Y_shuf[:, i:i+batch_size]
        m = Xb.shape[1]
        
        # Forward
        Z = W @ Xb + b
        A = sigmoid(Z)
        loss = -(1/m) * np.sum(Yb * np.log(A + 1e-8) + (1 - Yb) * np.log(1 - A + 1e-8))
        epoch_loss += loss
        
        # Backward
        dZ = A - Yb
        dW = (1/m) * dZ @ Xb.T
        db = (1/m) * np.sum(dZ)
        
        # Update
        W -= lr * dW
        b -= lr * db
    
    losses.append(epoch_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Cumulative Loss: {epoch_loss:.4f}")

print(f"\nFinal accuracy: {np.mean((sigmoid(W @ X + b) > 0.5) == Y) * 100:.1f}%")
